In [ ]:
import os                            
import json                          
import re                            
import numpy as np                   
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
import albumentations as A
import pytorch_lightning as pl
from transformers import (
    DonutProcessor,                  
    VisionEncoderDecoderModel,       
    VisionEncoderDecoderConfig,      
    get_linear_schedule_with_warmup  
)
from torch.optim import AdamW
from torch.utils.data import random_split
from pytorch_lightning.callbacks import (
    EarlyStopping, 
    ModelCheckpoint, 
    LearningRateMonitor,
    StochasticWeightAveraging
)

In [ ]:
import os
import json
import numpy as np
import albumentations as A
from PIL import Image
from torch.utils.data import Dataset

class DonutSROIEDataset(Dataset):
    def __init__(self, img_dir, ent_dir, processor, max_length=768, split="train"):
        """
        Custom Dataset for SROIE receipt parsing with Donut model.
        Args:
            img_dir: Directory containing receipt images.
            ent_dir: Directory containing ground truth text files (JSON or key:value format).
            processor: DonutProcessor for image and text tokenization.
            max_length: Maximum token sequence length for the generator.
        """
        self.img_dir = img_dir
        self.ent_dir = ent_dir
        self.processor = processor
        self.max_length = max_length
        self.split = split
        self.filenames = [f for f in os.listdir(img_dir) if f.lower().endswith(('.jpg', '.png', '.jpeg'))]

        # Heavy augmentations for real-world scanned receipts (noise, rotation, perspective)
        self.transform_real = A.Compose([
            A.OneOf([
                A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=5, border_mode=0, p=0.7),
                A.Perspective(scale=(0.02, 0.05), p=0.6),
            ], p=0.8),
            A.OneOf([
                A.Sharpen(alpha=(0.1, 0.3), p=0.5),
                A.ImageCompression(quality_lower=70, quality_upper=90, p=0.4),
                A.GaussNoise(var_limit=(10.0, 30.0), p=0.4),
            ], p=0.5),
            A.OneOf([
                A.CLAHE(clip_limit=3.0, tile_grid_size=(8, 8), p=0.6),
                A.RandomBrightnessContrast(brightness_limit=0.15, contrast_limit=0.2, p=0.6),
                A.Emboss(p=0.3),
            ], p=0.7),
            A.CoarseDropout(
                max_holes=10, 
                max_height=0.02, 
                max_width=0.04, 
                min_holes=5, 
                fill_value=255, 
                p=0.4
            ),
            A.ToGray(p=0.15),
        ])

        # Light augmentations for synthetic or clean digital images
        self.transform_synth = A.Compose([
            A.Perspective(scale=(0.01, 0.03), p=0.4), 
            A.CLAHE(clip_limit=2.0, p=0.5),           
            A.RandomBrightnessContrast(brightness_limit=0.1, contrast_limit=0.1, p=0.4),
            A.ToGray(p=0.1),
        ])

    def __len__(self):
        return len(self.filenames)

    def __getitem__(self, idx):
        filename = self.filenames[idx]
        file_id = os.path.splitext(filename)[0]
        image = Image.open(os.path.join(self.img_dir, filename)).convert('RGB')
        
        # Apply data augmentation only during training phase
        if self.split == "train":
            image_np = np.array(image)
            
            # Use specific transforms for synthetic/generated data
            if "ultra" in filename.lower() or "synth" in filename.lower():
                augmented = self.transform_synth(image=image_np)
            else:
                augmented = self.transform_real(image=image_np)
                
            image = Image.fromarray(augmented['image'])
        
        # Format the target text sequence using special Donut prompt tokens
        target_sequence = "<s_sroie>"
        ent_path = os.path.join(self.ent_dir, f"{file_id}.txt")
        data = {}

        # Parsing ground truth files (supports both JSON and raw colon-separated text)
        if os.path.exists(ent_path):
            with open(ent_path, 'r', encoding='utf-8') as f:
                content = f.read().strip()
                try:
                    data = json.loads(content)
                except json.JSONDecodeError:
                    for line in content.split('\n'):
                        if ":" in line:
                            k, v = line.split(":", 1)
                            data[k.strip().lower()] = v.strip()

            # Construct the ground truth string: <s_key>value</s_key>
            for key in ["company", "date", "address", "total", "cash", "change"]:
                value = data.get(key, "")
                target_sequence += f"<s_{key}>{value}</s_{key}>"
        
        target_sequence += "</s_sroie>"

        # Preprocess image into pixel values tensor
        pixel_values = self.processor(image, return_tensors="pt").pixel_values.squeeze()
        
        # Tokenize the target text sequence
        labels = self.processor.tokenizer(
            target_sequence,
            add_special_tokens=True,
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        ).input_ids.squeeze()

        # Set pad tokens to -100 so the loss function ignores them
        labels[labels == self.processor.tokenizer.pad_token_id] = -100
        
        return {"pixel_values": pixel_values, "labels": labels}

In [ ]:
class SROIEDataModule(pl.LightningDataModule):
    def __init__(self, train_img_dir, train_ent_dir, test_img_dir, test_ent_dir, processor, batch_size=1):
        super().__init__()
        self.train_img_dir = train_img_dir
        self.train_ent_dir = train_ent_dir
        self.test_img_dir = test_img_dir
        self.test_ent_dir = test_ent_dir
        
        self.processor = processor
        self.batch_size = batch_size

    def setup(self, stage=None):
        self.train_ds = DonutSROIEDataset(
            self.train_img_dir, 
            self.train_ent_dir, 
            self.processor, 
            split="train"
        )
        
        self.val_ds = DonutSROIEDataset(
            self.test_img_dir, 
            self.test_ent_dir, 
            self.processor, 
            split="val"
        )

    def train_dataloader(self):
        return DataLoader(
            self.train_ds, 
            batch_size=self.batch_size, 
            shuffle=True, 
            num_workers=2 
        )

    def val_dataloader(self):
        return DataLoader(
            self.val_ds, 
            batch_size=self.batch_size, 
            shuffle=False, 
            num_workers=2
        )

In [ ]:
import pytorch_lightning as pl
from transformers import VisionEncoderDecoderModel, AdamW, get_linear_schedule_with_warmup

class DonutFullModel(pl.LightningModule):
    def __init__(self, processor, lr=2e-6):
        """
        PyTorch Lightning Module for the Donut model.
        Args:
            processor: DonutProcessor (contains image processor and tokenizer).
            lr: Base learning rate for the optimizer.
        """
        super().__init__()
        # Save hyperparameters except the complex processor object
        self.save_hyperparameters(ignore=['processor'])
        self.processor = processor
        self.lr = lr
        
        # Initialize the VisionEncoderDecoderModel from pre-trained weights
        self.model = VisionEncoderDecoderModel.from_pretrained("naver-clova-ix/donut-base")
        
        # Resize decoder embeddings to match the new vocabulary size (including special task tokens)
        self.model.decoder.resize_token_embeddings(len(self.processor.tokenizer))
        
        # Configure model parameters for the specific SROIE extraction task
        self.model.config.pad_token_id = self.processor.tokenizer.pad_token_id
        self.model.config.decoder_start_token_id = self.processor.tokenizer.convert_tokens_to_ids('<s_sroie>')
        
        # Enable gradient checkpointing to save VRAM at the cost of slight speed reduction
        self.model.gradient_checkpointing_enable()

    def forward(self, pixel_values, labels):
        """Standard forward pass for VisionEncoderDecoderModel"""
        return self.model(pixel_values, labels=labels)
    
    def training_step(self, batch, batch_idx):
        """Single training iteration: calculate loss and log it"""
        pixel_values = batch["pixel_values"]
        labels = batch["labels"]
        
        outputs = self(pixel_values, labels)
        loss = outputs.loss
        
        # Log training loss to progress bar and logger (on both step and epoch levels)
        self.log("train_loss", loss, prog_bar=True, on_step=True, on_epoch=True)
        return loss
    
    def validation_step(self, batch, batch_idx):
        """Validation iteration to monitor the model's performance on unseen data"""
        outputs = self(batch["pixel_values"], batch["labels"]) 
        val_loss = outputs.loss
        
        self.log('val_loss', val_loss, on_epoch=True, prog_bar=True)
        return val_loss
   
    def configure_optimizers(self):
        """Setup AdamW optimizer and a Linear Scheduler with Warmup"""
        optimizer = AdamW(
            self.parameters(), 
            lr=self.lr,
            weight_decay=0.05  # Applied to prevent overfitting
        )
        
        # Calculate total training steps for the scheduler
        try:
            total_steps = self.trainer.estimated_stepping_batches
        except Exception:
            # Fallback value if trainer is not fully initialized
            total_steps = 10000 
            
        # 10% of total steps used for linear warmup to stabilize training start
        warmup_steps = int(total_steps * 0.1)
        
        scheduler = get_linear_schedule_with_warmup(
            optimizer, 
            num_warmup_steps=warmup_steps, 
            num_training_steps=total_steps
        )
        
        return {
            "optimizer": optimizer,
            "lr_scheduler": {
                "scheduler": scheduler,
                "interval": "step", # Update learning rate after every batch
            },
        }

MODEL LAUNCH UNIT

In [ ]:
import pytorch_lightning as pl
from transformers import DonutProcessor

if __name__ == "__main__":
    # Initialize the base Donut processor from Hugging Face
    model_name = "naver-clova-ix/donut-base"
    processor = DonutProcessor.from_pretrained(model_name)

    # Set high-resolution input size for better document detail extraction
    # We disable long axis alignment to keep aspect ratios consistent
    processor.image_processor.size = {"height": 1280, "width": 960}
    processor.image_processor.do_align_long_axis = False 

    # Define custom task-specific tokens for the SROIE dataset fields
    new_tokens = [
        "<s_company>", "</s_company>", 
        "<s_date>", "</s_date>", 
        "<s_address>", "</s_address>", 
        "<s_total>", "</s_total>",
        "<s_sroie>", "</s_sroie>" 
    ]
    processor.tokenizer.add_tokens(new_tokens)

    # Initialize the Lightning Module with a specified learning rate
    model = DonutFullModel(processor=processor, lr=2e-5)

    # Data paths (Note: These should be updated relative to the repo root for local runs)
    base_path = "/kaggle/input/datasets/maxbegal/dataset/data"
    
    train_img = f"{base_path}/train/img"
    train_ent = f"{base_path}/train/entities"
    test_img = f"{base_path}/val/img"
    test_ent = f"{base_path}/val/entities"

    # Initialize DataModule for handling train/val splits and loaders
    dm = SROIEDataModule(
        train_img_dir=train_img,
        train_ent_dir=train_ent,
        test_img_dir=test_img,
        test_ent_dir=test_ent,
        processor=processor,
        batch_size=1  # Low batch size due to high-resolution images and VRAM limits
    )

    # Define training callbacks for experiment management and regularization
    callbacks = [
        # Stop training if validation loss stops improving to prevent overfitting
        EarlyStopping(
            monitor="val_loss", 
            patience=8, 
            mode="min"
        ),
        # Save only the best performing model checkpoint
        ModelCheckpoint(
            monitor="val_loss",
            dirpath="checkpoints",
            filename="best-donut-{epoch:02d}-{val_loss:.2f}",
            save_top_k=1,
            mode="min",
        ),
        # Log learning rate changes during the warmup/linear decay phases
        LearningRateMonitor(logging_interval="step"),
        
        # SWA for better generalization and more stable convergence
        StochasticWeightAveraging(swa_lrs=2e-6) 
    ]
    
    # Configure the PyTorch Lightning Trainer with optimization features
    trainer = pl.Trainer(
        accelerator="gpu",
        devices=1,
        precision="16-mixed",         # Use Mixed Precision (FP16) to speed up training and save memory
        accumulate_grad_batches=8,    # Gradient Accumulation: effectively mimics a larger batch size (1*8=8)
        max_epochs=20,
        gradient_clip_val=1.0,        # Clip gradients to prevent exploding gradient issues
        callbacks=callbacks,  
        log_every_n_steps=10
    )
   
    # Reinforce memory efficiency during training
    model.model.gradient_checkpointing_enable()

    print("Starting FULL Fine-tuning on SROIE dataset...")
    trainer.fit(model, datamodule=dm)

    # Save final artifacts (model weights and processor) for inference or deployment
    output_dir = "donut_sroie_final_model"
    model.model.save_pretrained(output_dir)
    processor.save_pretrained(output_dir)
    
    print(f"Training complete. Model and processor saved to '{output_dir}'.")

Loading weights:   0%|          | 0/484 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie decoder.model.decoder.embed_tokens.weight to decoder.lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
/tmp/ipykernel_55/3902684456.py:25: UserWarning: Argument(s) 'quality_lower, quality_upper' are not valid for transform ImageCompression
  A.ImageCompression(quality_lower=70, quality_upper=90, p=0.4),
/tmp/ipykernel_55/3902684456.py:26: UserWarning: Argument(s) 'var_limit' are not valid for transform GaussNoise
  A.GaussNoise(var_limit=(10.0, 30.0), p=0.

🚀 Запуск ПОЛНОГО обучения (Full Fine-tuning) на SROIE...


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
Loading `train_dataloader` to estimate number of stepping batches.


┏━━━┳━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━┳━━━━━━━┓
┃   ┃ Name  ┃ Type                      ┃ Params ┃ Mode ┃ FLOPs ┃
┡━━━╇━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━╇━━━━━━━┩
│ 0 │ model │ VisionEncoderDecoderModel │  201 M │ eval │     0 │
└───┴───────┴───────────────────────────┴────────┴──────┴───────┘

...

Output()

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/callbacks/stochastic_weight_avg.py:232: SWA is currently 
only supported every epoch. Found LRSchedulerConfig(scheduler=<torch.optim.lr_scheduler.LambdaLR object at 
0x7a6dc4572db0>, name=None, interval='step', frequency=1, reduce_on_plateau=False, monitor=None, strict=True)

Swapping scheduler `LambdaLR` for `SWALR`
`Trainer.fit` stopped: `max_epochs=20` reached.


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Обучение завершено. Модель и процессор сохранены в 'donut_sroie_final_model'.


In [ ]:
import os
import json
import torch
import re
import editdistance
from tqdm import tqdm
from PIL import Image
from transformers import DonutProcessor, VisionEncoderDecoderModel

def parse_answer(text, key):
    """
    Ultimate parser.
    First cleans the text from tokenization artifacts, then extracts the value.
    """
    pattern = rf"<s_{key}>(.*?)</s_{key}>"
    match = re.search(pattern, text, re.IGNORECASE)
    
    val = ""
    if match:
        val = match.group(1).strip()
    else:
        # Fallback logic if tags are malformed
        fallback_pattern = rf"s_{key}\s*(.*?)(?:s_|/s_{key}|$)"
        fallback_match = re.search(fallback_pattern, text, re.IGNORECASE)
        if fallback_match:
            val = fallback_match.group(1).strip()
        else:
            if text.lower().startswith(f"s{key}"):
                val = text[len(f"s{key}"):].strip()
            else:
                val = text.strip()

    # List of prefixes to remove from the beginning of the string
    prefixes_to_kill = [f"s_{key}", f"s{key}", key, "s_"]
    
    changed = True
    while changed:
        changed = False
        current_lower = val.lower()
        for prefix in prefixes_to_kill:
            if current_lower.startswith(prefix):
                val = val[len(prefix):].strip()
                current_lower = val.lower()
                changed = True
        
        if current_lower.startswith('s '):
            val = val[2:].strip()
            changed = True

    return val.strip()

def normalize_text(text):
    """
    Highly aggressive normalization for SROIE.
    Ignores differences in ampersands, special characters, and spaces.
    """
    if not text:
        return ""
    
    text = str(text).lower()
    
    # Standardize conjunctions
    text = text.replace("&amp;", "&")
    text = text.replace("and", "&") 
    
    # Remove all non-alphanumeric characters
    text = re.sub(r'[^a-z0-9]', '', text)
    
    # Remove all whitespace
    text = text.replace(" ", "")
    
    return text.strip()


def run_evaluation(model_path, test_img_dir, test_ent_dir):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    
    # Loading model and processor
    print(f"Loading model from {model_path}...")
    processor = DonutProcessor.from_pretrained(model_path)
    model = VisionEncoderDecoderModel.from_pretrained(model_path).to(device)
    model.eval()

    filenames = [f for f in os.listdir(test_img_dir) if f.lower().endswith(('.jpg', '.png', '.jpeg'))]
    metrics = {"company": 0, "date": 0, "address": 0, "total": 0, "full_match": 0}
    similarity_scores = {"company": [], "date": [], "address": [], "total": []}
    
    total_count = len(filenames)
    print(f"Starting inference on {total_count} images...")

    for filename in tqdm(filenames):
        file_id = os.path.splitext(filename)[0]
        img_path = os.path.join(test_img_dir, filename)
        ent_path = os.path.join(test_ent_dir, f"{file_id}.txt")

        # Load Ground Truth data
        gt_data = {}
        if os.path.exists(ent_path):
            with open(ent_path, 'r', encoding='utf-8') as f:
                try:
                    gt_data = json.load(f)
                except Exception:
                    # Fallback for plain text entity files
                    f.seek(0)
                    for line in f:
                        if ":" in line:
                            k, v = line.strip().split(":", 1)
                            gt_data[k.lower().strip()] = v.strip()

        try:
            image = Image.open(img_path).convert("RGB")
        except Exception as e:
            print(f"Error loading {filename}: {e}")
            continue

        # Prepare inputs for Donut
        pixel_values = processor(image, return_tensors="pt").pixel_values.to(device)

        task_prompt = "<s_sroie>"
        decoder_input_ids = processor.tokenizer(task_prompt, add_special_tokens=False, return_tensors="pt").input_ids.to(device)

        # Generate prediction
        with torch.no_grad():
            outputs = model.generate(
                pixel_values,
                decoder_input_ids=decoder_input_ids,
                max_length=768,
                num_beams=1,
                return_dict_in_generate=True,
            )

        prediction_text = processor.batch_decode(outputs.sequences)[0]
        prediction_text = prediction_text.replace(processor.tokenizer.eos_token, "").replace(processor.tokenizer.pad_token, "")
        
        pred_data = {}
        for k in ["company", "date", "address", "total"]:
            pred_data[k] = parse_answer(prediction_text, k)
        
        is_full_match = True
        for key in ["company", "date", "address", "total"]:
            p = normalize_text(pred_data.get(key, ""))
            g = normalize_text(gt_data.get(key, ""))

            # Calculate Normalized Edit Distance
            if g:
                dist = editdistance.eval(p, g)
                score = 1 - (dist / max(len(p), len(g))) if max(len(p), len(g)) > 0 else 0
                similarity_scores[key].append(score)
            else:
                similarity_scores[key].append(0.0)

            # Calculate Accuracy
            if p == g and g != "":
                metrics[key] += 1
            else:
                is_full_match = False
        
        if is_full_match:
            metrics["full_match"] += 1

    # Final reporting
    print("\n" + "="*50)
    print("RESULTS WITH NORMALIZATION:")
    for key in ["company", "date", "address", "total"]:
        acc = (metrics[key] / total_count) * 100
        avg_sim = (sum(similarity_scores[key]) / len(similarity_scores[key])) * 100 if similarity_scores[key] else 0
        print(f" - {key.capitalize():<10}: Acc: {acc:>6.2f}% | Avg Similarity: {avg_sim:>6.2f}%")
    
    print("-" * 50)
    print(f"Full Match: {(metrics['full_match'] / total_count) * 100:.2f}%")
    print("="*50)

if __name__ == "__main__":
    MODEL_PATH = "./donut_sroie_final_model"
    TEST_IMG = "/kaggle/input/datasets/maxbegal/dataset/data/test/img"
    TEST_ENT = "/kaggle/input/datasets/maxbegal/dataset/data/test/entities"
    
    if os.path.exists(MODEL_PATH):
        run_evaluation(MODEL_PATH, TEST_IMG, TEST_ENT)
    else:
        print(f"Model not found at path: {MODEL_PATH}")

📦 Загрузка модели из ./donut_sroie_final_model...


Loading weights:   0%|          | 0/483 [00:00<?, ?it/s]

🔍 Запуск инференса на 168 изображениях...


100%|██████████| 168/168 [02:48<00:00,  1.00s/it]


📊 РЕЗУЛЬТАТЫ С НОРМАЛИЗАЦИЕЙ:
 - Company   : Acc:  82.74% | Avg Similarity:  92.62%
 - Date      : Acc:  89.88% | Avg Similarity:  96.12%
 - Address   : Acc:  80.95% | Avg Similarity:  95.13%
 - Total     : Acc:  87.50% | Avg Similarity:  94.75%
--------------------------------------------------
✅ Full Match: 61.90%


VISUALIZATION OF INFERENCE ERRORS

In [ ]:
import os
import json
import torch
import re
from tqdm import tqdm
from PIL import Image, ImageDraw, ImageFont
from transformers import DonutProcessor, VisionEncoderDecoderModel

def parse_answer(text, key):
    pattern = rf"<s_{key}>(.*?)</s_{key}>"
    match = re.search(pattern, text, re.IGNORECASE)
    return match.group(1).strip() if match else ""

def normalize_text(text):
    if not text: return ""
    text = str(text).lower()
    text = re.sub(r'[^a-z0-9\s]', '', text)
    return " ".join(text.split())

def create_error_image(image, results, output_path):
    """
    Creates a collage: original image on the left, 
    text table with results on the right.
    """
    w, h = image.size
    panel_width = 500
    new_w = w + panel_width
    
    combined = Image.new('RGB', (new_w, h), (255, 255, 255))
    combined.paste(image, (0, 0))
    
    draw = ImageDraw.Draw(combined)
    
    try:
        font = ImageFont.truetype("/usr/share/fonts/truetype/liberation/LiberationSans-Regular.ttf", 20)
        bold_font = ImageFont.truetype("/usr/share/fonts/truetype/liberation/LiberationSans-Bold.ttf", 24)
    except:
        font = ImageFont.load_default()
        bold_font = font

    x_offset = w + 20
    y_offset = 30
    
    draw.text((x_offset, y_offset), "DONUT SROIE ANALYSIS", fill=(0, 0, 0), font=bold_font)
    y_offset += 50
    
    for key, data in results.items():
        p = data['p']
        g = data['g']
        is_ok = normalize_text(p) == normalize_text(g)
        color = (0, 128, 0) if is_ok else (200, 0, 0)
        status = "OK" if is_ok else "ERR"
        
        draw.text((x_offset, y_offset), f"FIELD: {key.upper()}", fill=(100, 100, 100), font=font)
        y_offset += 25
        
        draw.text((x_offset, y_offset), f"P: {p[:40]}", fill=(0, 0, 0), font=font)
        y_offset += 25
        
        draw.text((x_offset, y_offset), f"G: {g[:40]}", fill=(150, 0, 0) if not is_ok else (0, 0, 0), font=font)
        y_offset += 25
        
        draw.text((x_offset, y_offset), f"STATUS: {status}", fill=color, font=bold_font)
        y_offset += 45
        draw.line([(x_offset, y_offset), (new_w-20, y_offset)], fill=(200, 200, 200))
        y_offset += 20

    combined.save(output_path)

def run_visual_export(model_path, test_img_dir, test_ent_dir, output_dir="errors", limit=50):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
        print(f"Created folder: {output_dir}")

    # Load processor and model
    processor = DonutProcessor.from_pretrained(model_path)
    model = VisionEncoderDecoderModel.from_pretrained(model_path).to(device)
    model.eval()

    filenames = [f for f in os.listdir(test_img_dir) if f.lower().endswith(('.jpg', '.png'))][:limit]
    
    print(f"Starting export of {len(filenames)} images...")

    for filename in tqdm(filenames):
        file_id = os.path.splitext(filename)[0]
        img_path = os.path.join(test_img_dir, filename)
        ent_path = os.path.join(test_ent_dir, f"{file_id}.txt")

        # Load ground truth data
        gt_data = {}
        if os.path.exists(ent_path):
            with open(ent_path, 'r', encoding='utf-8') as f:
                try: gt_data = json.load(f)
                except:
                    f.seek(0)
                    for line in f:
                        if ":" in line:
                            k, v = line.strip().split(":", 1)
                            gt_data[k.lower().strip()] = v.strip()

        # Inference
        image = Image.open(img_path).convert("RGB")
        pixel_values = processor(image, return_tensors="pt").pixel_values.to(device)
        task_prompt = "<s_sroie>"
        decoder_input_ids = processor.tokenizer(task_prompt, add_special_tokens=False, return_tensors="pt").input_ids.to(device)

        with torch.no_grad():
            outputs = model.generate(
                pixel_values, 
                decoder_input_ids=decoder_input_ids, 
                max_length=512,
                return_dict_in_generate=True
            )

        prediction_raw = processor.batch_decode(outputs.sequences)[0]
        
        # Prepare data for visualization
        res_to_draw = {}
        for key in ["company", "date", "address", "total"]:
            res_to_draw[key] = {
                'p': parse_answer(prediction_raw, key),
                'g': gt_data.get(key, "")
            }
        
        out_name = f"result_{filename}"
        create_error_image(image, res_to_draw, os.path.join(output_dir, out_name))

    print(f"Done! Look for images in: {os.path.abspath(output_dir)}")

if __name__ == "__main__":
    # Specify your paths here
    MODEL_PATH = "./donut_sroie_final_model" 
    TEST_IMG = "/kaggle/input/datasets/maxbegal/dataset/data/test/img"
    TEST_ENT = "/kaggle/input/datasets/maxbegal/dataset/data/test/entities"
    
    # Executing visual export
    run_visual_export(MODEL_PATH, TEST_IMG, TEST_ENT, output_dir="/kaggle/working/errors", limit=50)

📁 Создана папка: /kaggle/working/errors


Loading weights:   0%|          | 0/483 [00:00<?, ?it/s]

🚀 Начинаем экспорт 50 изображений...


100%|██████████| 50/50 [00:53<00:00,  1.06s/it]

✅ Готово! Ищи картинки в папке: /kaggle/working/errors
